# C5 – Empfehlungen & Visualisierungen (25M)

**Voraussetzung:** B hat `results/models/als_25m_final` bereits gespeichert.

Outputs → `results/figures/`:
- `rmse_heatmap.png`
- `score_distribution.png`
- `example_recommendations.png`

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS, ALSModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F
from pyspark.sql.window import Window

FIGURES = Path("results/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

spark = (
    SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "100")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark bereit.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 21:36:40 WARN Utils: Your hostname, LT-RD-286, resolves to a loopback address: 127.0.1.1; using 192.168.0.234 instead (on interface wlp0s20f3)
26/06/09 21:36:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 21:36:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark bereit.


## Daten & Modell laden

In [ ]:
df_ratings = (
    spark.read
    .option("header", True).option("inferSchema", True)
    .csv("../data/ml-25m/ratings.csv")
    .withColumn("userId",  F.col("userId").cast("int"))
    .withColumn("movieId", F.col("movieId").cast("int"))
    .withColumn("rating",  F.col("rating").cast("float"))
)

df_movies = (
    spark.read
    .option("header", True).option("inferSchema", True)
    .csv("../data/ml-25m/movies.csv")
)

# Gleiche Filterung wie in B3 (Filme mit ≥ 50 Bewertungen)
popular = (
    df_ratings.groupBy("movieId")
    .agg(F.count("*").alias("n"))
    .filter("n >= 50")
    .select("movieId")
)
df_ratings = df_ratings.join(popular, on="movieId")

train_df, test_df = df_ratings.randomSplit([0.8, 0.2], seed=42)

best_model = ALSModel.load("../results/models/als_25m_final")
print(f"Modell geladen: rank={best_model.rank}")
print(f"Ratings: {df_ratings.count():,}")

26/06/09 21:37:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: results/models/als_25m_final/metadata.
java.io.FileNotFoundException: File results/models/als_25m_final/metadata does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/regiodata/DHBW/SEM4/Meene/gitalt/Meene/notebooks/results/models/als_25m_final/metadata. SQLSTATE: 42K03

---
## 1 – RMSE über Hyperparameter-Variation

Manueller Sweep über dieselben Werte wie B4 (rank × regParam).
Ergebnis wird als CSV gecacht – bei erneutem Ausführen wird es direkt geladen.

In [ ]:
METRICS_CSV = Path("results/cv_metrics.csv")

if METRICS_CSV.exists():
    results_df = pd.read_csv(METRICS_CSV)
    print(f"Metriken aus Cache geladen: {METRICS_CSV}")
else:
    evaluator = RegressionEvaluator(
        metricName="rmse", labelCol="rating", predictionCol="prediction"
    )
    ranks      = [10, 20, 50]
    reg_params = [0.01, 0.1, 1.0]
    rows = []

    for rank in ranks:
        for reg in reg_params:
            als = ALS(
                userCol="userId", itemCol="movieId", ratingCol="rating",
                rank=rank, regParam=reg, maxIter=10,
                coldStartStrategy="drop", seed=42
            )
            rmse = evaluator.evaluate(als.fit(train_df).transform(test_df))
            rows.append({"rank": rank, "regParam": reg, "RMSE": round(rmse, 4)})
            print(f"rank={rank:>2}  regParam={reg:.2f}  RMSE={rmse:.4f}")

    results_df = pd.DataFrame(rows)
    METRICS_CSV.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(METRICS_CSV, index=False)
    print(f"\nGespeichert: {METRICS_CSV}")

results_df

In [ ]:
pivot = results_df.pivot(index="regParam", columns="rank", values="RMSE")

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="YlOrRd_r", aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"rank={r}" for r in pivot.columns], fontsize=10)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"λ={r}" for r in pivot.index], fontsize=10)
ax.set_title("RMSE über Hyperparameter-Variation (MovieLens 25M)", pad=12)

vmin, vmax = pivot.values.min(), pivot.values.max()
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = "white" if (val - vmin) / (vmax - vmin) < 0.4 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=11, fontweight="bold", color=color)

fig.colorbar(im, ax=ax, label="RMSE", shrink=0.85)
fig.tight_layout()
fig.savefig(FIGURES / "rmse_heatmap.png", bbox_inches="tight")
plt.show()

best = results_df.loc[results_df["RMSE"].idxmin()]
print(f"Beste Kombination → rank={int(best['rank'])}, λ={best['regParam']}, RMSE={best['RMSE']}")

---
## 2 – Empfehlungs-Score-Verteilung

In [ ]:
# Top-10 für alle User (wird auch für Schritt 3 weiterverwendet)
all_recs = best_model.recommendForAllUsers(10)

scores_pd = (
    all_recs
    .withColumn("rec", F.explode("recommendations"))
    .select(F.col("rec.rating").alias("score"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(scores_pd["score"], bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Predicted Score")
ax.set_ylabel("Anzahl Empfehlungen")
ax.set_title("Verteilung der Empfehlungs-Scores (alle User, Top-10)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

median = scores_pd["score"].median()
ax.axvline(median, color="firebrick", linestyle="--", linewidth=1.2,
           label=f"Median = {median:.2f}")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "score_distribution.png", bbox_inches="tight")
plt.show()
print(f"Score-Bereich: {scores_pd['score'].min():.2f} – {scores_pd['score'].max():.2f}")

---
## 3 – 3 Beispiel-User auswählen & Empfehlungen visualisieren

Strategie: Je einen User mit Drama-, Action- und Comedy-Profil auswählen
(Lieblingsgenre = Genre mit den meisten 4+ -Bewertungen).

In [ ]:
user_genres = (
    df_ratings
    .filter(F.col("rating") >= 4.0)
    .join(df_movies.select("movieId", "genres"), on="movieId")
    .withColumn("genre", F.explode(F.split("genres", "\\|")))
    .filter(F.col("genre") != "(no genres listed)")
    .groupBy("userId", "genre")
    .count()
)

top_genre_per_user = (
    user_genres
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("userId").orderBy(F.desc("count"))
    ))
    .filter(F.col("rn") == 1)
    .filter(F.col("count") >= 20)   # min. 20 Hochbewertungen im Lieblingsgenre
    .select("userId", "genre", "count")
)

selected_users = {}
for genre in ["Drama", "Action", "Comedy"]:
    row = (
        top_genre_per_user
        .filter(F.col("genre") == genre)
        .orderBy(F.desc("count"))
        .first()
    )
    if row:
        selected_users[genre] = row["userId"]
        print(f"{genre:8s}  userId={row['userId']:>6}  ({row['count']} Hochbewertungen)")

In [ ]:
users_spark = spark.createDataFrame(
    [(uid,) for uid in selected_users.values()], ["userId"]
)

recs_3 = (
    best_model.recommendForUserSubset(users_spark, 10)
    .withColumn("rec", F.explode("recommendations"))
    .select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("score")
    )
    .join(df_movies.select("movieId", "title", "genres"), on="movieId")
    .withColumn("rank", F.row_number().over(
        Window.partitionBy("userId").orderBy(F.desc("score"))
    ))
    .orderBy("userId", "rank")
    .toPandas()
)

uid_to_genre = {v: k for k, v in selected_users.items()}
recs_3["Profil"] = recs_3["userId"].map(uid_to_genre)

# Tabelle in Konsole ausgeben
for genre, uid in selected_users.items():
    sub = recs_3[recs_3["userId"] == uid][["rank", "title", "genres", "score"]].copy()
    sub["score"] = sub["score"].round(3)
    sub = sub.reset_index(drop=True)
    print(f"\n{'='*60}")
    print(f"User {uid}  |  Profil: {genre}")
    print(f"{'='*60}")
    print(sub.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = {"Drama": "#4C72B0", "Action": "#C44E52", "Comedy": "#55A868"}

for ax, (genre, uid) in zip(axes, selected_users.items()):
    sub = recs_3[recs_3["userId"] == uid].sort_values("rank")
    titles = sub["title"].str[:40].tolist()
    scores = sub["score"].tolist()
    n = len(titles)

    bars = ax.barh(
        range(n), scores[::-1],
        color=colors.get(genre, "steelblue"), alpha=0.85
    )
    ax.set_yticks(range(n))
    ax.set_yticklabels(titles[::-1], fontsize=7.5)
    ax.set_xlabel("Predicted Score", fontsize=9)
    ax.set_title(f"User {uid}\nProfil: {genre}", fontsize=10, fontweight="bold")

    x_max = max(scores) * 1.12
    ax.set_xlim(0, x_max)
    for bar, score in zip(bars[::-1], scores):
        ax.text(
            bar.get_width() + x_max * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{score:.2f}", va="center", fontsize=7.5
        )

fig.suptitle("Top-10-Empfehlungen für 3 Beispiel-User (MovieLens 25M)",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "example_recommendations.png", bbox_inches="tight")
plt.show()
print(f"Alle Figures gespeichert in: {FIGURES.resolve()}")

In [ ]:
spark.stop()